# 第8回 量子AI (QML) — 演習: 三目並べソルバーを作る

このノートでは、盤面を入れたら一手を返す **三目並べソルバー** を一から作ります。

ソルバーの中心には **AI** があります。AI は「いまの盤面」を受け取って、9 マスそれぞれに「ここに打つ価値」のスコアを出す関数です。一番スコアの高いマスに打つ、というのが基本動作です。

```
盤面(9マス) ──▶ [ AI ] ──▶ 9マスのスコア ──▶ 一番高いマスに打つ
```

この AI を、まず **古典のニューラルネット**で作って動かし、次に **量子回路**やそれを古典で挟んだ **ハイブリッド**に差し替えていきます。枠 (盤面→AI→スコア→一手) は変えず、AI の中身だけ差し替えて強さを比べるのが狙いです。

**このノートの流れ**

1. 三目並べの盤を用意する
2. 盤面を AI が読める数値に変換する
3. 古典 NN の AI を作って自己対戦で学習させる
4. 強さを **Elo レーティング**で測る
5. **動かす → 壊す → 直す**: 学習率をいじって学習が壊れる/直るのを体験する
6. AI を量子・ハイブリッドに差し替える
7. いろいろな設定を試す / 自分の AI と対戦して遊ぶ

> **実行時間の目安**: 古典パートは各 20 秒前後。量子パートは1モデルあたり**数分**かかります (量子回路をシミュレーションするため)。上から順に実行してください (後のセルは前のセルで作った関数・変数を使います)。

各演習ブロックには `EX1`〜`EX10` の番号が振ってあります。講義スライドの右上にも同じ番号が出るので、対応を確認しながら進められます。

## 0. セットアップ

使うライブラリを読み込みます。Qiskit (量子回路) と PyTorch (ニューラルネット) が必要です。未インストールなら先頭のコメントを外して実行してください。

`seed_all()` は乱数の種を固定する関数です。学習や対戦には乱数が絡むので、種を固定すると毎回ほぼ同じ結果が再現できます。

In [ ]:
# !pip install qiskit qiskit-machine-learning torch matplotlib
import time, random, warnings, logging
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")                                  # 量子ライブラリの警告を抑制
logging.getLogger("qiskit_machine_learning").setLevel(logging.ERROR)

SEED = 0
def seed_all(s=SEED):
    # Python標準・NumPy・PyTorch の乱数すべてに同じ種を渡す = 結果を再現可能にする
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

seed_all()
print("torch", torch.__version__)

## 1. 量子 AI の部品を確認する &nbsp;`EX1`

量子の AI を作る前に、部品となる量子回路がどんなものか、数値だけ先に見ておきます。ここではまだ学習はしません。「どんな部品があって、サイズ感はどれくらいか」を掴むのが目的です。

量子の AI は大きく2つの回路でできています。

- **埋め込み回路 (Feature Map)**: 古典の数値 (盤面) を量子状態に「載せる」回路。
- **ansatz (アンザッツ)**: 調整可能なつまみ (パラメータ θ) を持つ回路。古典 NN の「重み」にあたるのがこの θ で、学習でこれを動かします。

まず ansatz の θ がいくつあるかを見ます。`reps` (層の繰り返し回数) と qubit 数で決まります。

In [ ]:
# ansatz の調整パラメータ θ の数を数える (reps=2 のとき)
#   RealAmplitudes  : Ry 回転 + CX もつれ層 でできた標準的な ansatz
#   EfficientSU2    : Ry と Rz の2種の回転を使うので θ が増える
from qiskit.circuit.library import real_amplitudes, efficient_su2

for n in (3, 8):                                # qubit 数を 3 と 8 で比べる
    ra = real_amplitudes(n, reps=2).num_parameters
    es = efficient_su2(n,   reps=2).num_parameters
    print(f"qubit={n}:  RealAmplitudes の θ = {ra:2d}   EfficientSU2 の θ = {es}")
# qubit が増えると θ も増える。EfficientSU2 は回転が2種類なぶん θ が多い。

In [ ]:
# 埋め込み回路 (Feature Map) を実際に描いてみる
#   ZFeatureMap  : 各 qubit に独立に位相を載せるだけ → qubit 同士の "もつれ" なし
#   ZZFeatureMap : qubit の対にも位相を載せる        → もつれあり (表現力は上がるが重い)
from qiskit.circuit.library import z_feature_map, zz_feature_map

print("ZFeatureMap (もつれなし):")
print(z_feature_map(3, reps=1).decompose().draw(output="text"))
print("\nZZFeatureMap (もつれあり):")
print(zz_feature_map(3, reps=1).decompose().draw(output="text"))
# もつれを足すほど表現力は上がるが回路は深くなる。三目並べでは軽い方を基本にする。

## 2. 環境: 三目並べの盤

ソルバーが相手をする盤を作ります。`TicTacToe` クラスが盤の状態と進行を管理します。

- `board`: 9 要素のリスト。各マスは `+1` (先手 O)、`−1` (後手 X)、`0` (空) のいずれか。
- `player`: いまの手番 (`+1` または `−1`)。一手打つたびに入れ替わります。
- `legal()`: まだ空いている (打てる) マスの番号リスト。
- `winner()`: 縦横斜めの 8 ラインを調べ、揃っていればその手番 (`+1`/`−1`)、なければ `0` を返す。
- `step(a)`: マス `a` に現在の手番の石を置き、(新しい盤, 報酬, 終了フラグ, 勝者) を返す。報酬は「直前に打った人」から見て、勝ったら `+1`、それ以外は `0` です。

In [ ]:
# 勝ちになる 8 ライン (縦3・横3・斜め2) のマス番号
WIN = [(0,1,2),(3,4,5),(6,7,8),    # 横
       (0,3,6),(1,4,7),(2,5,8),    # 縦
       (0,4,8),(2,4,6)]            # 斜め

class TicTacToe:
    def __init__(self):
        self.reset()
    def reset(self):
        self.board = [0]*9          # 全マス空
        self.player = 1             # 先手は +1
        return self.board
    def legal(self):
        # 空いている(=0)マスの番号を返す
        return [i for i in range(9) if self.board[i] == 0]
    def winner(self):
        # 8ラインのどれかが同じ非ゼロで揃っていれば、その手番が勝者
        for a, b, c in WIN:
            if self.board[a] != 0 and self.board[a] == self.board[b] == self.board[c]:
                return self.board[a]
        return 0                    # まだ勝者なし
    def step(self, a):
        self.board[a] = self.player           # 現在の手番の石を置く
        w = self.winner()
        done = (w != 0 or not self.legal())   # 勝者が出た or 空きマス無し で終局
        r = 1.0 if w != 0 else 0.0            # 直前に打った player から見た即時報酬
        self.player = -self.player            # 手番交代
        return self.board, r, done, w

env = TicTacToe()
print("初期の合法手:", env.legal())

## 3. 盤面を AI が読める形にする &nbsp;`EX2`

ニューラルネットは数値ベクトルしか扱えないので、盤面を 9 次元のベクトルに変換します。`+1 / −1 / 0` はそのまま使えますが、一工夫します。

三目並べは先手後手が入れ替わります。同じネットに先手のときも後手のときも判断させたいので、`turn_normalize=True` のときは盤面に手番 (`player`) を掛けて、**「自分の石を常に +1」**に揃えます。こうすると AI は「自分=+1・相手=−1」という一貫した見方で盤面を読めます。

(この正規化を切るとどうなるかは、あとの「試してみよ」で実験できます。)

In [ ]:
def encode(board, player, turn_normalize=True):
    v = np.array(board, dtype=np.float32)
    if turn_normalize:
        v = v * player          # player を掛けて「自分の石=+1・相手=-1」に揃える
    return torch.from_numpy(v)  # PyTorch テンソルに変換して返す

# 例: 先手(player=1)から見た盤面
print(encode([1, -1, 0,  0, 1, 0,  -1, 0, 0], player=1))

## 4. 古典の AI を作る &nbsp;`EX3`

まず AI を**古典の全結合ニューラルネット**で作ります。9 個の入力 (盤面) → 隠れ層 → 9 個の出力 (各マスのスコア) という単純な構造です。出力は、そのマスに打つ価値 (Q値) を表します。

`act()` は「AI に盤面を見せて、実際にどのマスに打つか」を決める関数です。基本は一番スコアの高い**合法手**を選びますが、確率 `eps` でわざとランダムに打ちます (これを ε-greedy と呼びます)。学習中にいろいろな手を試させ、同じ手ばかりに偏らないための工夫です。

In [ ]:
class ClassicalAI(nn.Module):
    """古典の全結合ニューラルネット。盤面9 → 隠れ層 → スコア9。"""
    def __init__(self, hidden=64):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(9, hidden),   # 入力9 → 隠れ層
            nn.ReLU(),              # 非線形性
            nn.Linear(hidden, 9),   # 隠れ層 → 出力9 (各マスのスコア)
        )
    def forward(self, x):
        return self.f(x)

def act(net, board, player, eps, turn_normalize=True):
    """AI に board を見せて、打つマスを1つ返す。確率 eps でランダム手 (探索)。"""
    legal = [i for i in range(9) if board[i] == 0]   # 打てるマスだけが候補
    if random.random() < eps:
        return random.choice(legal)                  # 探索: ランダムに打つ
    with torch.no_grad():                            # 推論だけなので勾配は不要
        q = net(encode(board, player, turn_normalize))
    # 合法手のうちスコア最大のマスを選ぶ
    return max(legal, key=lambda i: q[i].item())

## 5. 自己対戦で学習させる

AI を、**自分自身と対戦させながら**強くします (自己対戦)。1 ゲームを最後まで打たせ、その記録を使ってスコアを修正していきます。

考え方は強化学習の Q学習です。各手について「この手を打った後、最終的にどうなったか」を手がかりに、スコアを正解に近づけます。

- ゲームが終わった手: 実際の報酬 `r` をそのまま正解に。
- 途中の手: 「報酬 − γ × (相手番での最大スコア)」を正解に。相手の番のスコアは自分にとって不利なので**符号を反転**して使います (`γ` は将来をどれだけ重視するかの割引率)。

この「正解」と現在のスコアの差を縮めるよう、ネットの重みを少しずつ更新します。

In [ ]:
def train_selfplay(net, games=800, lr=1e-3, eps=0.2, gamma=0.95, turn_normalize=True):
    """net を games 回の自己対戦で学習させて返す。"""
    opt = torch.optim.Adam(net.parameters(), lr=lr)   # 最適化器 (lr=学習率)
    lossf = nn.SmoothL1Loss()                         # スコアの誤差の測り方

    for g in range(games):
        # --- 1ゲームを最後まで自己対戦し、各手を記録する ---
        env = TicTacToe(); transitions = []
        while True:
            b = list(env.board); p = env.player
            a = act(net, b, p, eps, turn_normalize)    # ε-greedy で一手選ぶ
            nb, r, done, w = env.step(a)
            transitions.append((b, p, a, r, list(nb), done))
            if done:
                break

        # --- 記録した各手のスコアを正解に近づける ---
        for (b, p, a, r, nb, done) in transitions:
            s = encode(b, p, turn_normalize); q = net(s)
            with torch.no_grad():
                if done:
                    target_a = r                       # 終局手: 報酬そのもの
                else:
                    # 次は相手番。相手の最大スコアは自分に不利 → 符号反転して使う
                    nq = net(encode(nb, -p, turn_normalize))
                    nlegal = [i for i in range(9) if nb[i] == 0]
                    target_a = r - gamma * max(nq[i].item() for i in nlegal)
            t = q.detach().clone(); t[a] = target_a    # 打ったマスだけ正解を差し替え
            opt.zero_grad(); lossf(q, t).backward(); opt.step()   # 重みを更新
    return net

## 6. 強さを Elo で測る &nbsp;`EX4`

「強くなった」と言うには、ものさしが要ります。ここでは**ランダムに打つ相手**と何度も対戦させ、その成績を **Elo レーティング**という1つの数値にまとめます。Elo はチェスや将棋でも使われる強さの指標で、基準を 1500 とし、勝つほど上がり負けるほど下がります。

- `play_vs_random()`: ランダム相手と指定回数対戦し、勝ち/分け/負けの割合を返す。先手後手を交互に入れ替えて公平に測ります。
- `elo_vs_random()`: 100 ゲームずつのまとまりで成績を見て、期待勝率とのズレから Elo を更新していきます (`K=32` は1回の更新の大きさ)。

数式: 期待勝率 `W = 1 / (10^((相手R − 自分R)/400) + 1)`、更新 `R' = R + K·(実得点 − 試合数·W)`。

In [ ]:
def play_vs_random(net, turn_normalize=True, games=200):
    """net をランダム方策と games 回対戦させ、(勝率, 分率, 負率) を返す。"""
    wins = draws = losses = 0
    for g in range(games):
        env = TicTacToe()
        agent = 1 if g % 2 == 0 else -1     # 先手後手を交互に担当して公平に
        while True:
            if env.player == agent:
                a = act(net, list(env.board), env.player, 0.0, turn_normalize)  # AI の手 (探索なし)
            else:
                a = random.choice(env.legal())                                  # 相手はランダム
            _, _, done, w = env.step(a)
            if done:
                if   w == agent: wins   += 1
                elif w == 0:     draws  += 1
                else:            losses += 1
                break
    return wins/games, draws/games, losses/games

def elo_vs_random(net, turn_normalize=True, blocks=5, per=100, K=32, R0=1500, Ropp=1500):
    """100ゲームずつ blocks 回測り、Elo を更新していく。最終 Elo と推移を返す。"""
    R = R0; hist = [R]
    for _ in range(blocks):
        wr, dr, _ = play_vs_random(net, turn_normalize, games=per)
        score = wr*per + dr*0.5*per                  # 勝ち=1点・分け=0.5点
        W = 1/(10**((Ropp - R)/400) + 1)             # 期待勝率
        R = R + K*(score - per*W); hist.append(R)    # 期待とのズレぶん R を動かす
    return R, hist

### 動かす: 古典ソルバーを動かしてみる &nbsp;`EX5`

健全な学習率 (`lr=1e-3`) で学習させ、ランダム相手の勝率と Elo を見ます (約 20 秒)。学習前は弱く、少し学習させると明らかに強くなる、という最初の手応えを確認します。

In [ ]:
t0 = time.time()

# まず「未学習」の AI の強さを測る (学習前のベースライン)
seed_all()
fresh = ClassicalAI()
wr0, dr0, ls0 = play_vs_random(fresh)
print(f"未学習: vs random  勝 {wr0:.2f} / 分 {dr0:.2f} / 負 {ls0:.2f}   (でたらめに近い)")

# 健全な学習率で 800 ゲーム自己対戦学習させ、もう一度測る
# (学習を毎回同じ条件から始めるため、ここで種をまき直す)
seed_all()
classical = train_selfplay(ClassicalAI(), games=800, lr=1e-3, turn_normalize=True)
wr, dr, ls = play_vs_random(classical)               # 成績を測る
R_classical, hist_c = elo_vs_random(classical)       # Elo を測る
print(f"学習後: vs random  勝 {wr:.2f} / 分 {dr:.2f} / 負 {ls:.2f}   Elo={R_classical:.0f}   ({time.time()-t0:.0f}s)")
# 未学習 約0.48 → 学習後 約0.86 (Elo 約1707)。弱い土台が、学習で強くなった。

## 7. 動かす → 壊す → 直す &nbsp;`EX6`

ここがこの演習の本題です。学習がうまくいくかは設定 (ハイパーパラメータ) に左右されます。中でも効くのが **学習率 `lr`** — 重みを1回にどれだけ大きく動かすかです。

`lr` を上げすぎると、修正が大きすぎて学習が**発散** (暴れて壊れる) します。下のセルは `lr=0.3` というわざと大きすぎる値で学習させ、Elo が大きく下がる (壊れる) のを見ます。そのあと `lr` を `1e-3` に戻して再学習すれば Elo は回復します — これが「直す」です。

「壊して、原因 (学習率) に気づき、直す」という流れそのものが学びです。

In [ ]:
seed_all()
# 学習率を上げすぎる → 学習が発散して弱いまま
broken = train_selfplay(ClassicalAI(), games=800, lr=0.3, turn_normalize=True)
wrb, drb, lsb = play_vs_random(broken)
R_broken, _ = elo_vs_random(broken)
print(f"壊したAI (lr=0.3): 勝 {wrb:.2f} / 分 {drb:.2f} / 負 {lsb:.2f}   Elo={R_broken:.0f}")
print(f"健全なAIとの Elo 差: {R_classical - R_broken:.0f}  (差が大きいほど壊れている)")

# --- 直す: 下のコメントを外すと、lr を戻して再学習し Elo が回復するのを確認できる ---
# fixed = train_selfplay(ClassicalAI(), games=800, lr=1e-3, turn_normalize=True)

## 8. AI を量子に差し替える: ハイブリッド &nbsp;`EX7`

ここまでの AI は丸ごと古典でした。これを、**一部だけ量子回路に置き換えた**ハイブリッドにします。丸ごと量子にすると回路が大きく扱いにくいので、古典の層で挟んで要所だけ量子にするのが実用的です (NISQ 時代の定石)。

```
盤面9 ─▶ 古典Linear(9→N) ─▶ 量子層(N qubit) ─▶ 古典Linear(→9) ─▶ スコア9
```

量子層の「読み出し方」は2通りあり、後段の古典 Linear の入力サイズが変わります。

- **Estimator**: 各 qubit の期待値を読む → 出力は `N` 次元。
- **Sampler**: 測定結果の確率分布を読む → 出力は `2ⁿ` 次元 (大きい)。

量子層を PyTorch の層として扱えるようにする糊が `TorchConnector` です。これで量子層と古典層をまとめて、いつもの自己対戦学習にかけられます。

> ⏱ **このパートは数分かかります** (量子回路をシミュレーションするため)。

In [ ]:
from qiskit.circuit.library import real_amplitudes, z_feature_map
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector

def build_quantum_layer(N=4, reps=1, mode="estimator"):
    """N qubit の量子層を作り、(PyTorch層, 出力次元) を返す。"""
    fm  = z_feature_map(N, reps=1)        # 載せる回路 (入力を量子状態に)
    anz = real_amplitudes(N, reps=reps)   # 処理回路 (調整つまみ θ)
    qc = QuantumCircuit(N)
    qc.compose(fm,  inplace=True)         # 載せる → 処理 の順につなぐ
    qc.compose(anz, inplace=True)
    if mode == "estimator":
        # 各 qubit の Z 期待値を読む → 出力 N 次元
        obs = [SparsePauliOp("I"*(N-1-i) + "Z" + "I"*i) for i in range(N)]
        qnn = EstimatorQNN(circuit=qc, observables=obs,
                           input_params=fm.parameters, weight_params=anz.parameters)
        out_dim = N
    else:
        # 測定の確率分布を読む → 出力 2^N 次元
        qnn = SamplerQNN(circuit=qc, input_params=fm.parameters, weight_params=anz.parameters)
        out_dim = 2**N
    return TorchConnector(qnn), out_dim   # TorchConnector で PyTorch 層として使える

class HybridAI(nn.Module):
    """古典Linear で挟んで要所だけ量子にしたハイブリッド AI。"""
    def __init__(self, N=4, mode="estimator"):
        super().__init__()
        self.pre  = nn.Linear(9, N)                    # 盤面9 → N に圧縮 (古典)
        self.q, out = build_quantum_layer(N=N, mode=mode)   # 量子層
        self.post = nn.Linear(out, 9)                  # 量子の出力 → スコア9 (古典)
    def forward(self, x):
        x = torch.tanh(self.pre(x)) * np.pi            # 量子の入力角度に収める
        x = self.q(x)                                  # 量子層を通す
        return torch.tanh(self.post(x))

# 読み出し方による「後段Linearの入力次元」と「総パラメータ数」の違いを確認
for mode in ("estimator", "sampler"):
    m = HybridAI(N=4, mode=mode)
    npar = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{mode:9s}: 後段Linearの入力 {m.post.in_features:2d}  / 総パラメータ {npar}")
# Estimator は出力が小さく軽い。Sampler は 2^4=16 次元で後段が大きくパラメータが増える。

### 動かす: ハイブリッド AI を小規模学習 &nbsp;`EX8`

ハイブリッド AI を少しだけ学習させて Elo を見ます。三目並べは小さい問題なので、玩具規模の学習では Elo は基準 1500 付近にとどまりがちです。古典をはっきり上回らせるには学習ゲーム数を増やす必要があります (そのぶん時間がかかります)。

「ハイブリッド > 古典 > 量子単体」という順位は元論文の結果で、ここではその傾向を小規模に再現するのが狙いです。数値そのものより、同じ枠で AI だけ替えて比べられること自体が要点です。

In [ ]:
seed_all()
t0 = time.time()
# 量子層を含むので学習は重い。ゲーム数は控えめ(120)に。
hybrid = train_selfplay(HybridAI(N=4, mode="estimator"), games=120, lr=0.05, turn_normalize=True)
wrh, drh, lsh = play_vs_random(hybrid)
R_hybrid, _ = elo_vs_random(hybrid, blocks=3)
print(f"ハイブリッドAI: 勝 {wrh:.2f} / 分 {drh:.2f} / 負 {lsh:.2f}   Elo={R_hybrid:.0f}   ({time.time()-t0:.0f}s)")
print("※ games を増やすと Elo は上がります (例: 500〜1000)")

## 9. 試してみよ &nbsp;`EX9`

強さを左右する要素は学習率のほかにもたくさんあります。下のテンプレで **1つだけ**変えて、Elo がどう動くか観察してください。1つずつ変えるのが、効き目を見極めるコツです。

- **手番正規化** `turn_normalize=False`: 「自分の石を +1 に揃える」工夫を切る (盤面の見せ方を変える)
- **探索** `eps` を 0.05 / 0.5 に: 学習中にランダム手を打つ割合
- **割引** `gamma` を 0.8 / 0.99 に: 将来の報酬をどれだけ重視するか
- **量子の読み出し** `mode="sampler"`: 後段が大きくなる
- **qubit 数** `N=2` / `N=6`: 増やすほど表現力は上がるが、勾配が平坦になって学習しにくくなる (Barren Plateau と呼ばれる量子特有の壁)

In [ ]:
# テンプレ: 1要素だけ変えて、健全な古典AIと比べる
seed_all()
trial = train_selfplay(ClassicalAI(), games=800, lr=1e-3, turn_normalize=False)  # ← ここを変える
wr, dr, ls = play_vs_random(trial, turn_normalize=False)
R_trial, _ = elo_vs_random(trial, turn_normalize=False)
print(f"試行: 勝 {wr:.2f} / 分 {dr:.2f} / 負 {ls:.2f}   Elo={R_trial:.0f}")
print(f"健全な古典AIとの差: {R_classical - R_trial:.0f}")

## 10. 遊ぶ: 自分がいじった AI と対戦する &nbsp;`EX10`

ここまで AI の強さを **Elo の数値**で見てきました。最後に、その AI と**実際に対戦**して、強さの差を自分の手で体感します。

- **壊した AI (`broken`)** には勝てるはずです。
- **直した (健全な) AI (`classical`)** には、なかなか勝てません。

これが「壊す → 直す」の体感版です。数値で見た Elo の差が、対戦の手応えとして分かります。

> ⚠ **三目並べは最善どうしだと必ず引き分け**になります。強い AI に「勝てない」は、多くの場合「**引き分けが続く**」という形で現れます。引き分けが続いたら、それがその AI の強さだと思ってください。弱い AI (壊した方) には隙ができて勝てます。

> 対戦は楽しいぶん時間を使います。授業中に時間が無ければ、この notebook を**持ち帰って**遊べます (任意)。

In [ ]:
def show_board(board):
    """盤面を表示する。空きマスにはマス番号(0-8)を出すので、どこに打てるか分かる。"""
    sym = {1: "O", -1: "X", 0: None}
    rows = []
    for r in range(3):
        cells = []
        for c in range(3):
            v = board[r*3 + c]
            cells.append(sym[v] if v != 0 else str(r*3 + c))   # 空きはマス番号を表示
        rows.append(" " + " | ".join(cells) + " ")
    print(("\n" + "-"*11 + "\n").join(rows))

def play_vs_human(ai, human_first=True, turn_normalize=True):
    """人間 vs 学習済み AI を1局プレイする。人間はマス番号 0-8 を入力。q で中断。"""
    env = TicTacToe()
    human = 1 if human_first else -1     # 人間が先手(+1=O)か後手(-1=X)か
    print(f"あなた = {'O (先手)' if human_first else 'X (後手)'} / AI = {'X' if human_first else 'O'}")
    print("打ちたいマス番号 (0-8) を入力。q で中断。\n")
    show_board(env.board); print()

    while True:
        if env.player == human:
            # --- 人間の手番: 不正な入力でも落ちないように確認する ---
            while True:
                raw = input(f"あなたの手 (空きマス {env.legal()}): ").strip()
                if raw.lower() == "q":
                    print("中断しました。"); return None
                if not raw.lstrip("-").isdigit():
                    print("  数字を入れてください。"); continue            # 非数値を弾く
                a = int(raw)
                if a not in env.legal():
                    print("  そこには打てません (埋まっている / 範囲外)。"); continue  # 不正マスを弾く
                break
        else:
            # --- AI の手番: 学習済み AI に最善手を選ばせる (探索なし eps=0) ---
            a = act(ai, list(env.board), env.player, eps=0.0, turn_normalize=turn_normalize)
            print(f"AI は {a} に打ちました。")

        _, _, done, w = env.step(a)
        print(); show_board(env.board); print()
        if done:
            if   w == human: print(">>> あなたの勝ち!")
            elif w == 0:     print(">>> 引き分け (この AI 相手なら上出来)。")
            else:            print(">>> AI の勝ち。")
            return w

相手を選んで対戦してください。まず **壊した AI (`broken`)** で勝てることを確かめ、次に **直した AI (`classical`)** に挑むと差が体感できます。

In [ ]:
# 対戦相手を選ぶ: broken(壊した) / classical(直した・健全) / hybrid(ハイブリッド)
#   ※ これらは上のセルで学習済みである必要があります (上から順に実行してください)
opponent = broken          # ← broken / classical / hybrid に書き換えて相手を選ぶ
tn = True                  # 学習時と同じ手番正規化の設定にそろえる

play_vs_human(opponent, human_first=True, turn_normalize=tn)

## まとめ

- 枠 (盤面 → AI → スコア → 一手) は変えず、**AI の中身だけ**古典 / 量子 / ハイブリッドに差し替えて、Elo で公平に比べられた。
- **学習率**のような1つの設定で学習は簡単に壊れ、戻せば回復する (動かす → 壊す → 直す)。
- 量子単体は玩具規模では基準 1500 付近にとどまりやすい。「ハイブリッド > 古典 > 量子単体」という傾向は、規模を上げるほどはっきり見えてくる。

数値そのものより、**「AI を変えると・設定を1つ変えると、強さがどう動くか」**を自分の手で確かめることが目的です。最後の対戦では、その差を Elo の数値だけでなく手応えとして体感できます。